# Phase 10: Exploratory Data Analysis & Statistical Hypothesis Testing
**Financial Transaction Risk & Operations Intelligence Platform**

This interactive notebook conducts non-parametric statistical hypothesis testing, empirical distribution analysis, Weight of Evidence (WoE) and Information Value (IV) ranking, and operational turnaround time profiling across 6,362,620 financial transactions and 5,000 companion investigation cases.

In [ ]:
import sys
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from financial_risk.config import DATA_PROCESSED_DIR, FIGURES_DIR
from financial_risk.eda import FinancialRiskEDA

eda = FinancialRiskEDA()
print("FinancialRiskEDA engine initialized.")

## 1. Portfolio Population & Class Imbalance Baseline

In [ ]:
df = eda.load_transactions_data()
total_tx = len(df)
fraud_tx = int((df['isFraud'] == 1).sum())
benign_tx = total_tx - fraud_tx
fraud_pct = (fraud_tx / total_tx) * 100.0
total_gtv = df['amount'].sum()
fraud_gtv = df.loc[df['isFraud'] == 1, 'amount'].sum()

print(f"Total Transactions : {total_tx:,}")
print(f"Total GTV          : ${total_gtv:,.2f} ($1.144 Trillion)")
print(f"Fraud Volume       : {fraud_tx:,} transactions ({fraud_pct:.4f}% prevalence)")
print(f"Fraud Loss         : ${fraud_gtv:,.2f} ($12.056 Billion)")
print(f"Mean Fraud Ticket  : ${df.loc[df['isFraud']==1, 'amount'].mean():,.2f}")
print(f"Mean Benign Ticket : ${df.loc[df['isFraud']==0, 'amount'].mean():,.2f} (8.24x disparity)")

## 2. Non-Parametric Continuous Hypothesis Testing: Mann-Whitney U & K-S Tests
Testing whether continuous transaction features (amount, balance deltas, discrepancies) differ significantly between Fraud and Benign classes without assuming normality.

In [ ]:
mw_res = eda.run_mann_whitney_tests(df)
ks_res = eda.run_kolmogorov_smirnov_tests(df)

rows = []
for col in mw_res:
    rows.append({
        "Feature": col,
        "MW U-Statistic": f"{mw_res[col]['u_statistic']:,.0f}",
        "p-value": f"{mw_res[col]['p_value']:.2e}",
        "Rank-Biserial r": mw_res[col]['rank_biserial_correlation'],
        "MW Effect Size": mw_res[col]['effect_size_magnitude'],
        "KS D-Statistic": ks_res.get(col, {}).get('ks_statistic_d', 'N/A'),
        "KS Separation": ks_res.get(col, {}).get('separation_power', 'N/A')
    })
pd.DataFrame(rows)

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'fig_01_amount_distribution.png')))

## 3. Channel Vulnerability & Chi-Square Independence Testing
Testing independence between categorical attributes (type, night hours, merchant flag, balance depletion) and fraud occurrence using Cramér's V.

In [ ]:
chi2_res = eda.run_chi_square_tests(df)
chi2_rows = []
for col in chi2_res:
    chi2_rows.append({
        "Categorical Feature": col,
        "Chi2 Stat": f"{chi2_res[col]['chi2_statistic']:,.2f}",
        "Degrees of Freedom": chi2_res[col]['degrees_of_freedom'],
        "p-value": f"{chi2_res[col]['p_value']:.2e}",
        "Cramér's V": chi2_res[col]['cramers_v'],
        "Association": chi2_res[col]['association_strength']
    })
pd.DataFrame(chi2_rows)

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'fig_02_channel_vulnerability.png')))

## 4. Diurnal Diachronic Risk Profile (24-Hour Cycle)

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'fig_03_diurnal_hourly_risk.png')))

## 5. Weight of Evidence (WoE) & Information Value (IV) Ranking

In [ ]:
iv_res = eda.compute_information_values(df)
iv_rows = [
    {"Feature": k, "Information Value (IV)": v["information_value"], "Predictive Strength": v["predictive_tier"]}
    for k, v in iv_res.items()
]
pd.DataFrame(iv_rows)

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'fig_06_woe_iv_ranking.png')))

## 6. Spearman Rank Correlation Matrix & Multicollinearity

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'fig_04_correlation_heatmap.png')))

## 7. Investigation Operations & SLA Governance Analysis

In [ ]:
ops_summary = eda.compute_investigation_metrics()
print(f"Total Investigation Cases : {ops_summary['total_cases']:,}")
print(f"Closed Cases             : {ops_summary['closed_count']:,}")
print(f"Open Backlog             : {ops_summary['open_backlog']:,}")
print(f"SLA Breaches             : {ops_summary['sla_breach_count']:,} ({ops_summary['overall_sla_breach_rate_pct']}%) ")
print(f"Mean Turnaround Time     : {ops_summary['turnaround_time_hours']['mean']} hours")
print(f"Median P50 Turnaround    : {ops_summary['turnaround_time_hours']['median_p50']} hours")
print(f"P90 Turnaround Time      : {ops_summary['turnaround_time_hours']['p90']} hours")

display(Image(filename=str(FIGURES_DIR / 'fig_05_investigation_operations.png')))

## 8. Strategic Modeling Synthesis for Phase 11
1. **Cost-Sensitive Objective**: Due to extreme 0.1291% imbalance and $1.47M mean fraud loss, standard accuracy or binary cross-entropy will fail. Phase 11 will optimize cost-weighted loss matrices (penalizing false negatives significantly more than false positives).
2. **Dominant Signals**: Models must prioritize `orig_depletion_complete` (IV=3.58), `orig_balance_discrepancy` (IV=5.40, KS=0.81), and `is_transfer_or_cashout` (IV=7.96).
3. **Nocturnal Multiplier**: Dynamic thresholding will adjust decision boundaries during off-hours (00-06h) to leverage the 172.8x risk multiplier.